# 02. Vector Manipulation

Coming off [NB1](./01_introduction.ipynb): you've seen `upsert` and `query`. That's the happy path. Now we pick up speed.

**This is the walk.** Same shape of work, but we'll push on every side of it: filtering by metadata, updating records, deleting by id and by filter, and splitting data across namespaces. Still no embeddings, still no LLM. The toy vectors stick around because the goal is to get fluent with the API, not to get tripped up by an embedding call.

## The surface area we're poking at

```
          ┌─────────────────────────────────────────┐
          │                Pinecone                 │
          │  ┌─────────────────────────────────┐    │
          │  │          one index              │    │
          │  │  ┌─────────────────────────┐    │    │
          │  │  │       vectors           │    │    │
          │  │  │   upsert   fetch        │    │    │
          │  │  │   query    update       │    │    │
          │  │  │   delete                │    │    │
          │  │  └─────────────────────────┘    │    │
          │  │                                  │    │
          │  │  namespaces: isolated slices    │    │
          │  │                                  │    │
          │  └─────────────────────────────────┘    │
          └─────────────────────────────────────────┘
```

By the end you'll have touched every op in that inner box at least once.

## A richer toy dataset

NB1 had five hand-written 4-d vectors. That was fine for *one* query. For filtering and namespaces we need something with variety, so this notebook uses a seeded "movie catalog": 20 random-but-deterministic 16-dimensional vectors with metadata you can actually filter on (`title`, `genre`, `year`, `rating`).

It lives in `helpers.make_toy_movies`. Seeded, so the numbers are the same every run.

In [26]:
from helpers import load_env, get_pinecone_client, ensure_index, make_toy_movies, format_matches_table

cfg = load_env()
pc = get_pinecone_client(cfg)

movies = make_toy_movies(seed=42)
print(f"{len(movies)} movies, dim={len(movies[0]['values'])}")
print(f"Example: {movies[0]['metadata']}")

20 movies, dim=16
Example: {'title': 'The Vanishing Signal', 'genre': 'thriller', 'year': 2014, 'rating': 7.8}


In [27]:
INDEX_NAME = "rag-unpacked-movies"

index = ensure_index(
    pc,
    name=INDEX_NAME,
    dimension=16,
    metric="cosine",
    cloud=cfg.pinecone_cloud,
    region=cfg.pinecone_region,
)

index.upsert(vectors=movies)
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '246',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 23 Apr 2026 19:46:01 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 16,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 20},
                'batch_2024': {'vector_count': 10},
                'batch_2025': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 40,
 'vector_type': 'dense'}


## Fetch by id

`query` is for "find similar things". `fetch` is for "I already know the id, give me back what's stored". Use it to double-check an upsert landed the way you expected, or to pull a known set of ids from some other system.

In [28]:
fetched = index.fetch(ids=["m_03", "m_07", "m_12"])
for mid, vec in fetched["vectors"].items():
    print(f"  {mid}: {vec['metadata']}")

  m_03: {'genre': 'drama', 'rating': 7.2, 'title': 'Paper Kingdom', 'year': 2012}
  m_07: {'genre': 'thriller', 'rating': 7.5, 'title': 'The Glass Hour', 'year': 2016}
  m_12: {'genre': 'sci-fi', 'rating': 6.8, 'title': 'Static Gardens', 'year': 2013}


## Similarity query, bigger this time

Same call as NB1, but we ask for five results and format them as a table.

The `include_metadata=True` flag matters. Without it, Pinecone only sends back ids and scores. Most of the time you want the metadata back too, otherwise you have to make a second fetch to find out what the ids mean.

In [29]:
target_vec = next(m["values"] for m in movies if m["id"] == "m_06")

result = index.query(
    vector=target_vec,
    top_k=5,
    include_metadata=True,
)
format_matches_table(result)

,id,score,genre,rating,title,year
0,m_06,1.0000,sci-fi,8.6,Neon Cartography,2023
1,m_17,0.3890,horror,7.7,Feral Static,2023
2,m_09,0.3619,sci-fi,7.1,Moonwake,2015
3,m_13,0.3126,horror,7.4,The Cold Room,2019
4,m_15,0.2607,horror,6.5,Hollow Tide,2016


## Filtering by metadata

Similarity alone is blind to everything except geometry. Real apps almost always need to mix "find similar things" with "but only where some condition holds". Thriller genre only. Year before 2018. This tenant's data.

Pinecone's `filter=` argument handles that. It runs *before* ranking, so you don't waste work retrieving candidates that you'd just throw away client-side.

Operators you'll use:

| Operator | Meaning | Example |
|----------|---------|---------|
| `$eq` or the bare key | equals | `{"genre": "thriller"}` |
| `$ne` | not equals | `{"genre": {"$ne": "horror"}}` |
| `$gt`, `$gte`, `$lt`, `$lte` | numeric comparisons | `{"year": {"$lt": 2018}}` |
| `$in` | value in list | `{"genre": {"$in": ["sci-fi", "thriller"]}}` |
| `$nin` | value not in list | `{"year": {"$nin": [2020, 2021]}}` |
| `$exists` | key present | `{"rating": {"$exists": True}}` |

Top-level keys are combined with an implicit AND. For OR you wrap things in `$or`.

In [30]:
# Thrillers only
result = index.query(
    vector=target_vec,
    top_k=5,
    filter={"genre": "thriller"},
    include_metadata=True,
)
format_matches_table(result)

,id,score,genre,rating,title,year
0,m_01,0.2193,thriller,7.8,The Vanishing Signal,2014
1,m_04,0.1226,thriller,8.3,Last Train to Osaka,2017
2,m_11,-0.0610,thriller,8.0,Northbound,2024
3,m_07,-0.0892,thriller,7.5,The Glass Hour,2016
4,m_18,-0.1913,thriller,7.0,The Depot,2015


In [31]:
# Thrillers released before 2018
result = index.query(
    vector=target_vec,
    top_k=5,
    filter={"genre": "thriller", "year": {"$lt": 2018}},
    include_metadata=True,
)
format_matches_table(result)

,id,score,genre,rating,title,year
0,m_01,0.2193,thriller,7.8,The Vanishing Signal,2014
1,m_04,0.1226,thriller,8.3,Last Train to Osaka,2017
2,m_07,-0.0892,thriller,7.5,The Glass Hour,2016
3,m_18,-0.1913,thriller,7.0,The Depot,2015


In [32]:
# Sci-fi OR thriller, any year
result = index.query(
    vector=target_vec,
    top_k=5,
    filter={"genre": {"$in": ["sci-fi", "thriller"]}},
    include_metadata=True,
)
format_matches_table(result)

,id,score,genre,rating,title,year
0,m_06,1.0000,sci-fi,8.6,Neon Cartography,2023
1,m_09,0.3619,sci-fi,7.1,Moonwake,2015
2,m_14,0.2330,sci-fi,8.4,Soft Machines,2022
3,m_01,0.2193,thriller,7.8,The Vanishing Signal,2014
4,m_20,0.1380,sci-fi,8.2,Iron Archive,2025


## Updating records

Three flavours to pick from:

1. `index.update(id=..., values=...)` swaps the vector, leaves metadata alone.
2. `index.update(id=..., set_metadata={...})` merges the metadata keys you give it, doesn't touch the vector.
3. Just call `upsert` again with the same id. Identical final state, and often the simplest thing.

Rule of thumb: use `update` when you're changing one of values or metadata. Re-upsert when you're changing both.

In [33]:
# Swap the vector for m_07 with a new random unit vector
import numpy as np

rng = np.random.default_rng(0)
new_vec = rng.normal(size=16).astype(np.float32)
new_vec /= np.linalg.norm(new_vec)

index.update(id="m_07", values=new_vec.tolist())

fetched = index.fetch(ids=["m_07"])
print("m_07 first 4 dims after update:", fetched["vectors"]["m_07"]["values"][:4])
print("m_07 metadata still there:", fetched["vectors"]["m_07"]["metadata"])

m_07 first 4 dims after update: [0.0342215374, -0.0359566025, 0.174311697, 0.0285519529]
m_07 metadata still there: {'genre': 'thriller', 'rating': 7.5, 'title': 'The Glass Hour', 'year': 2016}


In [34]:
# Bump m_07's rating without touching the vector or other metadata
index.update(id="m_07", set_metadata={"rating": 9.1})

fetched = index.fetch(ids=["m_07"])
print(fetched["vectors"]["m_07"]["metadata"])

{'genre': 'thriller', 'rating': 9.1, 'title': 'The Glass Hour', 'year': 2016}


## Deleting records

Three variants, in order of how scared you should be:

- **By id.** `index.delete(ids=[...])`. Targeted, safe.
- **By filter.** `index.delete(filter={...})`. Handy for purging all of a document's chunks, and easy to nuke way more than you meant. Always run `describe_index_stats()` before and after.
- **Everything in a namespace.** `index.delete(delete_all=True, namespace="...")`. Clears the slate.

In [35]:
# Targeted delete
print("before:", index.describe_index_stats()["total_vector_count"])

index.delete(ids=["m_18", "m_19"])

print("after:", index.describe_index_stats()["total_vector_count"])

before: 40
after: 38


In [36]:
# Filter delete: purge all horror movies
print("before:", index.describe_index_stats()["total_vector_count"])

index.delete(filter={"genre": "horror"})

print("after:", index.describe_index_stats()["total_vector_count"])

before: 38
after: 35


## Namespaces

A namespace is a logical partition inside one index. A query scoped to one namespace never sees another namespace's rows. Three places this shows up a lot:

- **Multi-tenancy.** One namespace per customer, cleanly isolated, without paying for N separate indexes.
- **A/B testing a chunking strategy.** Load the same corpus two different ways into two namespaces, compare retrieval quality.
- **Staging vs production.** Load tomorrow's data into `staging`, query `prod`, swap when you're happy.

Every data op (`upsert`, `query`, `fetch`, `update`, `delete`) takes an optional `namespace=`. Leave it off and you're hitting the default namespace.

```
         ┌─────────── one index ───────────┐
         │                                 │
         │  ┌──────────┐  ┌──────────┐     │
         │  │ batch_24 │  │ batch_25 │ ... │
         │  │  10 vecs │  │  10 vecs │     │
         │  └──────────┘  └──────────┘     │
         │                                 │
         └─────────────────────────────────┘
```

In [37]:
# Make two batches and drop each into its own namespace
import numpy as np
rng = np.random.default_rng(1)

def random_unit_vectors(n, dim=16):
    out = rng.normal(size=(n, dim)).astype(np.float32)
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    return out

batch_2024 = [
    {"id": f"y24_{i:02d}", "values": vec.tolist(), "metadata": {"batch": "2024", "n": i}}
    for i, vec in enumerate(random_unit_vectors(10))
]
batch_2025 = [
    {"id": f"y25_{i:02d}", "values": vec.tolist(), "metadata": {"batch": "2025", "n": i}}
    for i, vec in enumerate(random_unit_vectors(10))
]

index.upsert(vectors=batch_2024, namespace="batch_2024")
index.upsert(vectors=batch_2025, namespace="batch_2025")
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '246',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 23 Apr 2026 19:46:04 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 16,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 15},
                'batch_2024': {'vector_count': 10},
                'batch_2025': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 35,
 'vector_type': 'dense'}


In [38]:
# Query scoped to batch_2025. A batch_2024 vector can never win.
query_vec = batch_2024[0]["values"]
result = index.query(
    vector=query_vec,
    top_k=3,
    namespace="batch_2025",
    include_metadata=True,
)
format_matches_table(result)

,id,score,batch,n
0,y25_05,0.4845,2025,5
1,y25_09,0.4678,2025,9
2,y25_07,0.4269,2025,7


In [39]:
# Per-namespace counts confirm the isolation
stats = index.describe_index_stats()
for ns, info in stats["namespaces"].items():
    print(f"  namespace={ns!r:20s}  count={info['vector_count']}")

  namespace='batch_2024'          count=10
  namespace='batch_2025'          count=10
  namespace='__default__'         count=15


## Recap

| Operation | Call | Use when |
|-----------|------|----------|
| Create index | `pc.create_index(name, dim, metric, spec)` or `ensure_index` | Once per dataset |
| Check existence | `pc.has_index(name)` | Guard before `create_index` or `delete_index` |
| Inspect | `pc.list_indexes`, `pc.describe_index`, `index.describe_index_stats` | Debugging, monitoring |
| Upsert | `index.upsert(vectors=[...])` | Adding or replacing vectors |
| Fetch | `index.fetch(ids=[...])` | You already know the id |
| Query | `index.query(vector=..., top_k=..., filter=..., namespace=...)` | Similarity search |
| Update values | `index.update(id=..., values=...)` | Change only the vector |
| Update metadata | `index.update(id=..., set_metadata={...})` | Merge metadata keys |
| Delete | `index.delete(ids=[...])`, `delete(filter=...)`, `delete(delete_all=True, ...)` | Remove records |
| Delete index | `pc.delete_index(name)` | Cleanup |

## Where this goes next

[03. Performance & RAG](./03_performance_and_rag.ipynb) is the run. Toys are gone, we load a real corpus, embed it with OpenAI, measure batched vs async ingest, and wire the whole thing into a small Q&A bot.

In [40]:
# Optional cleanup. Uncomment to run.
# if pc.has_index(INDEX_NAME):
#     pc.delete_index(INDEX_NAME)
#     print(f"Deleted {INDEX_NAME!r}.")